In [164]:
import pandas as pd
import numpy as np

In [165]:
movies = pd.read_csv('tmdb_5000_movies.csv')
credits = pd.read_csv('tmdb_5000_credits.csv')

In [166]:
movies.sample(1)

,budget,genres,homepage,id,keywords,original_language,original_title,overview,popularity,production_companies,production_countries,release_date,revenue,runtime,spoken_languages,status,tagline,title,vote_average,vote_count
2490,16000000,"[{""id"": 12, ""name"": ""Adventure""}, {""id"": 28, ""...",NaN,8009,"[{""id"": 242, ""name"": ""new york""}, {""id"": 388, ...",en,Highlander,He fought his first battle on the Scottish Hig...,29.253833,"[{""name"": ""Davis-Panzer Productions"", ""id"": 27...","[{""iso_3166_1"": ""GB"", ""name"": ""United Kingdom""...",1986-03-07,5900000,116.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,There can be only one.,Highlander,6.8,624


In [167]:
credits.sample(1)

,movie_id,title,cast,crew
959,9624,On Deadly Ground,"[{""cast_id"": 1, ""character"": ""Forrest Taft"", ""...","[{""credit_id"": ""52fe4513c3a36847f80baed9"", ""de..."


In [168]:
movies = movies.merge(credits , on='title')   #merger both datasets on col title

In [169]:
movies.info() # taking cols = genres, id, keywords, title, cast, crew, overview -- help in our analysis
# language ignored beacuse mostly are english here
# id taken to later use to fetch their posters
# we don't want numerical data more here

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4809 entries, 0 to 4808
Data columns (total 23 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   budget                4809 non-null   int64  
 1   genres                4809 non-null   object 
 2   homepage              1713 non-null   object 
 3   id                    4809 non-null   int64  
 4   keywords              4809 non-null   object 
 5   original_language     4809 non-null   object 
 6   original_title        4809 non-null   object 
 7   overview              4806 non-null   object 
 8   popularity            4809 non-null   float64
 9   production_companies  4809 non-null   object 
 10  production_countries  4809 non-null   object 
 11  release_date          4808 non-null   object 
 12  revenue               4809 non-null   int64  
 13  runtime               4807 non-null   float64
 14  spoken_languages      4809 non-null   object 
 15  status               

In [170]:
movies = movies[['movie_id','original_title','overview','genres','keywords','cast','crew']]

In [171]:
movies.isnull().sum()

movie_id          0
original_title    0
overview          3
genres            0
keywords          0
cast              0
crew              0
dtype: int64

In [172]:
movies.dropna(inplace=True)           #inplace = true -- modifies in original df 

In [173]:
movies.duplicated().sum()

0

In [174]:
movies.iloc[0].genres

'[{"id": 28, "name": "Action"}, {"id": 12, "name": "Adventure"}, {"id": 14, "name": "Fantasy"}, {"id": 878, "name": "Science Fiction"}]'

In [175]:
import ast

In [176]:
def convert(text):
    L = []
    for i in ast.literal_eval(text):        # to convert string to object
        L.append(i['name']) 
    return L 

In [177]:
# Apply the robust convert function
movies['genres'] = movies['genres'].apply(convert)
movies['keywords'] = movies['keywords'].apply(convert)


In [178]:
def convert3(text):
    L = []
    counter = 0
    for i in ast.literal_eval(text):
        if counter < 3:
            L.append(i['name'])
        counter+=1
    return L 

In [179]:
movies['cast'] = movies['cast'].apply(convert3)    # taken top 3 chars

In [180]:
movies['cast']

0        [Sam Worthington, Zoe Saldana, Sigourney Weaver]
1           [Johnny Depp, Orlando Bloom, Keira Knightley]
2            [Daniel Craig, Christoph Waltz, Léa Seydoux]
3            [Christian Bale, Michael Caine, Gary Oldman]
4          [Taylor Kitsch, Lynn Collins, Samantha Morton]
                              ...                        
4804    [Carlos Gallardo, Jaime de Hoyos, Peter Marqua...
4805         [Edward Burns, Kerry Bishé, Marsha Dietlein]
4806           [Eric Mabius, Kristin Booth, Crystal Lowe]
4807            [Daniel Henney, Eliza Coupe, Bill Paxton]
4808    [Drew Barrymore, Brian Herzlinger, Corey Feldman]
Name: cast, Length: 4806, dtype: object

In [181]:
movies.iloc[0].crew

'[{"credit_id": "52fe48009251416c750aca23", "department": "Editing", "gender": 0, "id": 1721, "job": "Editor", "name": "Stephen E. Rivkin"}, {"credit_id": "539c47ecc3a36810e3001f87", "department": "Art", "gender": 2, "id": 496, "job": "Production Design", "name": "Rick Carter"}, {"credit_id": "54491c89c3a3680fb4001cf7", "department": "Sound", "gender": 0, "id": 900, "job": "Sound Designer", "name": "Christopher Boyes"}, {"credit_id": "54491cb70e0a267480001bd0", "department": "Sound", "gender": 0, "id": 900, "job": "Supervising Sound Editor", "name": "Christopher Boyes"}, {"credit_id": "539c4a4cc3a36810c9002101", "department": "Production", "gender": 1, "id": 1262, "job": "Casting", "name": "Mali Finn"}, {"credit_id": "5544ee3b925141499f0008fc", "department": "Sound", "gender": 2, "id": 1729, "job": "Original Music Composer", "name": "James Horner"}, {"credit_id": "52fe48009251416c750ac9c3", "department": "Directing", "gender": 2, "id": 2710, "job": "Director", "name": "James Cameron"},

In [182]:
def fetch_director(text):
    L = []
    for i in ast.literal_eval(text):
        if i['job'] == 'Director':
            L.append(i['name'])
    return L 

In [183]:
movies['crew'] = movies['crew'].apply(fetch_director)

In [184]:
movies.head()

,movie_id,original_title,overview,genres,keywords,cast,crew
0,19995,Avatar,"In the 22nd century, a paraplegic Marine is di...","[Action, Adventure, Fantasy, Science Fiction]","[culture clash, future, space war, space colon...","[Sam Worthington, Zoe Saldana, Sigourney Weaver]",[James Cameron]
1,285,Pirates of the Caribbean: At World's End,"Captain Barbossa, long believed to be dead, ha...","[Adventure, Fantasy, Action]","[ocean, drug abuse, exotic island, east india ...","[Johnny Depp, Orlando Bloom, Keira Knightley]",[Gore Verbinski]
2,206647,Spectre,A cryptic message from Bond’s past sends him o...,"[Action, Adventure, Crime]","[spy, based on novel, secret agent, sequel, mi...","[Daniel Craig, Christoph Waltz, Léa Seydoux]",[Sam Mendes]
3,49026,The Dark Knight Rises,Following the death of District Attorney Harve...,"[Action, Crime, Drama, Thriller]","[dc comics, crime fighter, terrorist, secret i...","[Christian Bale, Michael Caine, Gary Oldman]",[Christopher Nolan]
4,49529,John Carter,"John Carter is a war-weary, former military ca...","[Action, Adventure, Science Fiction]","[based on novel, mars, medallion, space travel...","[Taylor Kitsch, Lynn Collins, Samantha Morton]",[Andrew Stanton]


In [185]:
movies['overview'][0]

'In the 22nd century, a paraplegic Marine is dispatched to the moon Pandora on a unique mission, but becomes torn between following orders and protecting an alien civilization.'

In [186]:
# converting this string to list  # to finally club all cols
movies['overview'] = movies['overview'].apply(lambda x:x.split())

In [187]:
movies.head()

,movie_id,original_title,overview,genres,keywords,cast,crew
0,19995,Avatar,"[In, the, 22nd, century,, a, paraplegic, Marin...","[Action, Adventure, Fantasy, Science Fiction]","[culture clash, future, space war, space colon...","[Sam Worthington, Zoe Saldana, Sigourney Weaver]",[James Cameron]
1,285,Pirates of the Caribbean: At World's End,"[Captain, Barbossa,, long, believed, to, be, d...","[Adventure, Fantasy, Action]","[ocean, drug abuse, exotic island, east india ...","[Johnny Depp, Orlando Bloom, Keira Knightley]",[Gore Verbinski]
2,206647,Spectre,"[A, cryptic, message, from, Bond’s, past, send...","[Action, Adventure, Crime]","[spy, based on novel, secret agent, sequel, mi...","[Daniel Craig, Christoph Waltz, Léa Seydoux]",[Sam Mendes]
3,49026,The Dark Knight Rises,"[Following, the, death, of, District, Attorney...","[Action, Crime, Drama, Thriller]","[dc comics, crime fighter, terrorist, secret i...","[Christian Bale, Michael Caine, Gary Oldman]",[Christopher Nolan]
4,49529,John Carter,"[John, Carter, is, a, war-weary,, former, mili...","[Action, Adventure, Science Fiction]","[based on novel, mars, medallion, space travel...","[Taylor Kitsch, Lynn Collins, Samantha Morton]",[Andrew Stanton]


In [188]:
def collapse(L):
    L1 = []
    for i in L:
        L1.append(i.replace(" ",""))     #saari spaces hata rhe like name sam mendes -- both will count as diff entities so remove space to make one
    return L1

In [189]:
movies['genres'] = movies['genres'].apply(collapse)
movies['keywords'] = movies['keywords'].apply(collapse)
movies['overview'] = movies['overview'].apply(collapse)
movies['cast'] = movies['cast'].apply(collapse)
movies['crew'] = movies['crew'].apply(collapse)

In [190]:
movies.head()

,movie_id,original_title,overview,genres,keywords,cast,crew
0,19995,Avatar,"[In, the, 22nd, century,, a, paraplegic, Marin...","[Action, Adventure, Fantasy, ScienceFiction]","[cultureclash, future, spacewar, spacecolony, ...","[SamWorthington, ZoeSaldana, SigourneyWeaver]",[JamesCameron]
1,285,Pirates of the Caribbean: At World's End,"[Captain, Barbossa,, long, believed, to, be, d...","[Adventure, Fantasy, Action]","[ocean, drugabuse, exoticisland, eastindiatrad...","[JohnnyDepp, OrlandoBloom, KeiraKnightley]",[GoreVerbinski]
2,206647,Spectre,"[A, cryptic, message, from, Bond’s, past, send...","[Action, Adventure, Crime]","[spy, basedonnovel, secretagent, sequel, mi6, ...","[DanielCraig, ChristophWaltz, LéaSeydoux]",[SamMendes]
3,49026,The Dark Knight Rises,"[Following, the, death, of, District, Attorney...","[Action, Crime, Drama, Thriller]","[dccomics, crimefighter, terrorist, secretiden...","[ChristianBale, MichaelCaine, GaryOldman]",[ChristopherNolan]
4,49529,John Carter,"[John, Carter, is, a, war-weary,, former, mili...","[Action, Adventure, ScienceFiction]","[basedonnovel, mars, medallion, spacetravel, p...","[TaylorKitsch, LynnCollins, SamanthaMorton]",[AndrewStanton]


In [191]:
movies['tags'] = movies['overview'] + movies['genres'] + movies['keywords'] + movies['cast'] + movies['crew']

In [192]:
movies.head()

,movie_id,original_title,overview,genres,keywords,cast,crew,tags
0,19995,Avatar,"[In, the, 22nd, century,, a, paraplegic, Marin...","[Action, Adventure, Fantasy, ScienceFiction]","[cultureclash, future, spacewar, spacecolony, ...","[SamWorthington, ZoeSaldana, SigourneyWeaver]",[JamesCameron],"[In, the, 22nd, century,, a, paraplegic, Marin..."
1,285,Pirates of the Caribbean: At World's End,"[Captain, Barbossa,, long, believed, to, be, d...","[Adventure, Fantasy, Action]","[ocean, drugabuse, exoticisland, eastindiatrad...","[JohnnyDepp, OrlandoBloom, KeiraKnightley]",[GoreVerbinski],"[Captain, Barbossa,, long, believed, to, be, d..."
2,206647,Spectre,"[A, cryptic, message, from, Bond’s, past, send...","[Action, Adventure, Crime]","[spy, basedonnovel, secretagent, sequel, mi6, ...","[DanielCraig, ChristophWaltz, LéaSeydoux]",[SamMendes],"[A, cryptic, message, from, Bond’s, past, send..."
3,49026,The Dark Knight Rises,"[Following, the, death, of, District, Attorney...","[Action, Crime, Drama, Thriller]","[dccomics, crimefighter, terrorist, secretiden...","[ChristianBale, MichaelCaine, GaryOldman]",[ChristopherNolan],"[Following, the, death, of, District, Attorney..."
4,49529,John Carter,"[John, Carter, is, a, war-weary,, former, mili...","[Action, Adventure, ScienceFiction]","[basedonnovel, mars, medallion, spacetravel, p...","[TaylorKitsch, LynnCollins, SamanthaMorton]",[AndrewStanton],"[John, Carter, is, a, war-weary,, former, mili..."


In [193]:
new_df = movies[['movie_id','original_title','tags']]

In [194]:
new_df.head()

,movie_id,original_title,tags
0,19995,Avatar,"[In, the, 22nd, century,, a, paraplegic, Marin..."
1,285,Pirates of the Caribbean: At World's End,"[Captain, Barbossa,, long, believed, to, be, d..."
2,206647,Spectre,"[A, cryptic, message, from, Bond’s, past, send..."
3,49026,The Dark Knight Rises,"[Following, the, death, of, District, Attorney..."
4,49529,John Carter,"[John, Carter, is, a, war-weary,, former, mili..."


In [196]:
new_df['tags'] = new_df['tags'].apply(lambda x: " ".join(x))  # convert list to string
new_df.head()

C:\Users\gnish\AppData\Local\Temp\ipykernel_21308\3040553394.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  new_df['tags'] = new_df['tags'].apply(lambda x: " ".join(x))  # convert list to string


,movie_id,original_title,tags
0,19995,Avatar,"In the 22nd century, a paraplegic Marine is di..."
1,285,Pirates of the Caribbean: At World's End,"Captain Barbossa, long believed to be dead, ha..."
2,206647,Spectre,A cryptic message from Bond’s past sends him o...
3,49026,The Dark Knight Rises,Following the death of District Attorney Harve...
4,49529,John Carter,"John Carter is a war-weary, former military ca..."


In [198]:
new_df['tags'] = new_df['tags'].apply(lambda x: x.lower())  # recommmended

C:\Users\gnish\AppData\Local\Temp\ipykernel_21308\4164960838.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  new_df['tags'] = new_df['tags'].apply(lambda x: x.lower())  # recommmended


In [201]:
new_df.head()

,movie_id,original_title,tags
0,19995,Avatar,"in the 22nd century, a paraplegic marine is di..."
1,285,Pirates of the Caribbean: At World's End,"captain barbossa, long believed to be dead, ha..."
2,206647,Spectre,a cryptic message from bond’s past sends him o...
3,49026,The Dark Knight Rises,following the death of district attorney harve...
4,49529,John Carter,"john carter is a war-weary, former military ca..."


In [220]:
import nltk

In [224]:
from nltk.stem.porter import PorterStemmer
ps = PorterStemmer()

In [225]:
def stem(text):
    l=[]
    for i in text.split(): #string to list
        l.append(ps.stem(i))

    return " ".join(l)
        

In [226]:
new_df['tags'] = new_df['tags'].apply(stem)

C:\Users\gnish\AppData\Local\Temp\ipykernel_21308\3213734980.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  new_df['tags'] = new_df['tags'].apply(stem)


In [227]:
from sklearn.feature_extraction.text import CountVectorizer
cv = CountVectorizer(max_features=4000 , stop_words='english')

In [228]:
vector = cv.fit_transform(new_df['tags']).toarray()

In [229]:
print(cv.get_feature_names_out().tolist())

['000', '007', '10', '100', '11', '12', '13', '14', '15', '16', '17', '18', '18th', '19', '1910', '1930', '1940', '1950', '1950s', '1960', '1960s', '1970', '1970s', '1980', '1990', '19th', '19thcenturi', '20', '200', '20th', '24', '25', '30', '3d', '40', '50', '60', '70', 'aaron', 'aaroneckhart', 'abandon', 'abduct', 'abigailbreslin', 'abil', 'abl', 'aboard', 'abov', 'abus', 'academi', 'accept', 'access', 'accid', 'accident', 'accompani', 'accomplish', 'account', 'accus', 'ace', 'achiev', 'act', 'action', 'activ', 'activist', 'actor', 'actress', 'actual', 'adam', 'adamsandl', 'adamshankman', 'adapt', 'add', 'addict', 'adjust', 'admir', 'admit', 'adolesc', 'adopt', 'ador', 'adrienbrodi', 'adult', 'adulteri', 'adulthood', 'advanc', 'adventur', 'adventure', 'advertis', 'advic', 'advis', 'affair', 'affect', 'afghanistan', 'africa', 'african', 'africanamerican', 'aftercreditssting', 'afterlif', 'aftermath', 'ag', 'age', 'agediffer', 'agency', 'agenda', 'agent', 'aggress', 'ago', 'agre', 'ah

In [233]:
from sklearn.metrics.pairwise import cosine_similarity # cosine similarity is 1/angle between them -- euclidean distance fail on large dimensions

In [234]:
similarity = cosine_similarity(vector)

In [235]:
similarity

array([[1.        , 0.08964215, 0.09258201, ..., 0.04829453, 0.        ,
        0.        ],
       [0.08964215, 1.        , 0.06454972, ..., 0.02525381, 0.        ,
        0.02727724],
       [0.09258201, 0.06454972, 1.        , ..., 0.02608203, 0.        ,
        0.        ],
       ...,
       [0.04829453, 0.02525381, 0.02608203, ..., 1.        , 0.04307305,
        0.04408667],
       [0.        , 0.        , 0.        , ..., 0.04307305, 1.        ,
        0.09304842],
       [0.        , 0.02727724, 0.        , ..., 0.04408667, 0.09304842,
        1.        ]])

In [251]:
list(enumerate(similarity[0]))

[(0, 1.0),
 (1, 0.08964214570007951),
 (2, 0.09258200997725513),
 (3, 0.07807200583588264),
 (4, 0.2038588765750502),
 (5, 0.11501092655705902),
 (6, 0.04259177099999598),
 (7, 0.15289415743128765),
 (8, 0.06506000486323554),
 (9, 0.1019294382875251),
 (10, 0.10826639239215334),
 (11, 0.09968895725584535),
 (12, 0.0994490316197694),
 (13, 0.04733810683311508),
 (14, 0.13710212427677043),
 (15, 0.07171371656006362),
 (16, 0.0857142857142857),
 (17, 0.14675987714106858),
 (18, 0.10030135678562992),
 (19, 0.09200874124564722),
 (20, 0.06121333677828399),
 (21, 0.11595420713048966),
 (22, 0.07559289460184544),
 (23, 0.09258200997725513),
 (24, 0.0563436169819011),
 (25, 0.03827795011754763),
 (26, 0.16035674514745463),
 (27, 0.20149814827784024),
 (28, 0.12344267996967351),
 (29, 0.06837634587578276),
 (30, 0.06965451903188581),
 (31, 0.16903085094570333),
 (32, 0.08877935232369503),
 (33, 0.10141851056742199),
 (34, 0.0),
 (35, 0.1057361065275362),
 (36, 0.1825741858350554),
 (37, 0.08964

In [262]:
def recommend(movie):
    index = new_df[new_df['original_title'] == movie].index[0]   # index of any movie can be taken
    distances = sorted(list(enumerate(similarity[index])),reverse=True,key = lambda x: x[1]) 
    # sorting the similarity array based on distance 
    # rev will do in desc order
    # enumerate will assign index to each similarity(0,1 : 1, 0.008) (movie x ka movie wih index 0 ke saath similarity and so on)
    # lambda function to sort movie based on x[1] , second entry i.e similarity not indexes
    for i in distances[1:6]:
        print(new_df.iloc[i[0]].original_title)
        

In [263]:
new_df[new_df['original_title'] == 'Batman Begins'].index[0]       # how index of any movie can be taken

119

In [264]:
recommend('Avatar')

Aliens vs Predator: Requiem
Independence Day
Falcon Rising
Jupiter Ascending
Titan A.E.


In [255]:
pickle.dump(new_df,open('movie_list.pkl','wb'))
pickle.dump(similarity,open('similarity.pkl','wb'))

In [252]:
import pickle